# 01 — 探索性数据分析（EDA）

**目标**：证明规范化 schema 能直接支撑有意义的业务查询，并用 6 张图把「结构」与「洞察」连接起来。

所有查询都在 **PostgreSQL 规范化模式** 上写标准 SQL（本地回退为同构 SQLite），通过外键 join 维度表展开分析。


In [ ]:
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

BASE = Path.cwd().parent
DB = BASE / 'data' / 'challenge_01.sqlite'
conn = sqlite3.connect(DB)

def q(sql):
    return pd.read_sql_query(sql, conn)


## 2.1 部门营收分布

`line_items → products → categories → departments` 四级 join，是「宽表」无法一键完成、而规范模式一条 SQL 即可得到的聚合。


In [ ]:
rev_dept = q('''
SELECT d.department_name AS department, SUM(li.line_subtotal) AS revenue
FROM line_items li
JOIN products p ON p.product_upc = li.product_upc
JOIN categories c ON c.category_id = p.category_id
JOIN departments d ON d.department_id = c.department_id
GROUP BY d.department_name ORDER BY revenue DESC
''')
rev_dept


In [ ]:
rev_dept.plot.barh(x='department', y='revenue', legend=False, figsize=(8,4))
plt.title('Revenue by department')
plt.tight_layout(); plt.show()


![部门营收](../reports/figures/fig1_department_revenue.png)

**洞察**：Grocery 是绝对主力（≈$174k，占 34%），Wellness / Meat-Seafood / Dairy / Produce 构成第二梯队，Bakery 明显偏弱。


## 2.2 月度营收趋势与促销占比


In [ ]:
monthly = q('''
SELECT strftime('%m', r.sale_datetime) AS month,
       SUM(li.line_subtotal) AS revenue,
       SUM(CASE WHEN li.promo_id IS NOT NULL THEN li.line_subtotal ELSE 0 END) AS promo_rev
FROM line_items li JOIN receipts r ON r.receipt_id = li.receipt_id
GROUP BY month ORDER BY month
''')
monthly['promo_share'] = 100*monthly.promo_rev/monthly.revenue
monthly


![月度趋势](../reports/figures/fig2_monthly_trend.png)

**洞察**：月营收在 $40k–$45k 间窄幅波动（无强季节性）；促销成交占比稳定在 8%–11%，Q4（10、11 月）略升。


## 2.3 门店 × 客群表现


In [ ]:
store_seg = q('''
SELECT s.store_name AS store, r.customer_segment AS segment,
       SUM(li.line_subtotal) AS revenue
FROM receipts r JOIN stores s ON s.store_id=r.store_id
JOIN line_items li ON li.receipt_id=r.receipt_id
GROUP BY s.store_name, r.customer_segment
''').pivot(index='store', columns='segment', values='revenue')
store_seg


![门店×客群](../reports/figures/fig3_store_segment.png)

**洞察**：三店体量几乎一致（各 ≈$170k），会员贡献稳定高于非会员，是标准的「三店均衡」格局。


## 2.4 促销 lift（每行平均营收 vs 数量）


In [ ]:
lift = q('''
SELECT COALESCE(pr.promo_type,'NO_PROMO') AS promo_type,
       AVG(li.unit_price_effective) AS avg_price,
       SUM(li.quantity)*1.0/COUNT(*) AS avg_qty,
       SUM(li.line_subtotal)*1.0/COUNT(*) AS avg_line_rev
FROM line_items li LEFT JOIN promotions pr ON pr.promo_id=li.promo_id
GROUP BY pr.promo_type
''')
lift


![促销 lift](../reports/figures/fig4_promo_lift.png)

**洞察**：BOGO50 把每行平均数量从 1.29 拉到 **2.16**、行均营收提高到 $11.9（最高），是「提量」型促销；PCT_OFF / LOYALTY_PCT 是「降价换量」型，行均单价更低。


## 2.5 会员 vs 非会员


![会员占比](../reports/figures/fig5_member_mix.png)

**洞察**：会员贡献约 59% 营收（客单价相近、但会员交易笔数更多），是非会员的 1.4 倍。


## 2.6 Top 商品


In [ ]:
top = q('''
SELECT p.product_name, SUM(li.line_subtotal) AS revenue
FROM line_items li JOIN products p ON p.product_upc=li.product_upc
GROUP BY p.product_name ORDER BY revenue DESC LIMIT 10
''')
top


![Top 商品](../reports/figures/fig6_top_products.png)

**洞察**：头号单品 Sun Valley Basmati Rice（$29k）领先，Top10 集中在 Grocery 主食与高单价 Wellness / Meat-Seafood。


## 小结

规范模式的价值在此充分体现：一张票行事实表 + 多张维度表，通过外键即可在**不复制、不漂移**的前提下回答「哪个部门/门店/商品/促销/客群贡献最大」这一整类问题。
